# Marketplace Dynamics and Behavioral Patterns in Brazilian E-Commerce
*Exploring relationships across pricing, logistics, purchasing patterns, and customer activity.*

## 1. Introduction

This notebook continues directly from *Data Preparation and Merging*, using the cleaned and enriched `order_details` created during that stage.

The original Olist data consisted of nine interconnected relational datasets spanning customers, sellers, products, payments, reviews, geolocation records, order items, order information, and product category translations. These datasets existed across multiple granularities and required careful integration to create a unified analytical view. Preparing the data involved establishing relationships across operational, geographic, and transactional sources while preserving the structure needed for later analysis.

The resulting dataset enters this analysis at the order-item level and contains 113,425 observations. It has been cleaned, validated, and enriched with engineered features related to pricing, logistics, seller characteristics, and marketplace activity.

This exploratory analysis follows questions that emerged throughout preparation and early investigation. Rather than approaching the data with a predefined question, the analysis follows the structure of the marketplace itself. The Olist platform connects pricing, logistics, seller behavior, and customer experience in ways that are difficult to untangle cleanly. Patterns in one area have a way of pointing toward questions in another. The analysis begins broadly and narrows gradually, letting the data surface what matters.

## 2. Environment Setup

This section initializes the notebook environment, including library imports, display settings, and plotting conventions used throughout the analysis.

### 2.1 Libraries

In [22]:
# File handling
import os
import sys

# Data manipulation and utilities
import pandas as pd
import numpy as np
import math
from IPython.display import display, Markdown

# Visualization
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import seaborn as sns

# Statistical tests
from scipy.stats import kruskal, mannwhitneyu, chi2_contingency
from statsmodels.stats.multitest import multipletests

# Warnings 
import warnings

### 2.2 Environment Configuration

In [23]:
sys.path.append("..")
from config import PROCESSED_DATA

### 2.3 Display Settings

In [24]:
# DataFrame display
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 120)

# Plot theme defaults
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.labelsize"] = 10

# Suppress non-critical warnings for cleaner notebook output
warnings.filterwarnings("ignore")

### 2.4 Color Conventions

In [25]:
# Olist brand colors
NAVY    = "#1A1F5E"
TEAL    = "#2EC4A0"
SKY     = "#4BBFE0"

# Additional colors
CORAL   = "#F4845F"
YELLOW  = "#FFD166"

# Binary palettes
PALETTE_ONTIME = {
    True: TEAL,
    False: CORAL
    }
PALETTE_SELLER_TYPE = {
    "specialist": SKY,
    "generalist": TEAL
    }

# Review score palette (1–5)
PALETTE_REVIEW = {
    1: CORAL,
    2: YELLOW,
    3: TEAL,
    4: SKY,
    5: NAVY
}

# Heatmap / sequential
CMAP_SEQ = sns.light_palette(TEAL, as_cmap=True)
CMAP_DIV = sns.diverging_palette(357, 170, s=75, l=50, as_cmap=True)

# Categorical palette (up to 10 categories)
CATEGORY_PALETTE = sns.color_palette("muted", 10)

# Single-variable plots
SINGLE_COLOR = SKY

## 3. Unified Marketplace Overview
The analysis begins by loading the cleaned and enriched `order_details` dataset created during the data preparation stage.

In [26]:
df = pd.read_parquet(os.path.join(PROCESSED_DATA, "order_details.parquet"))

In [27]:
# Dataset summary
n_rows = f"{df.shape[0]:,}"
n_features = df.shape[1]

purchase_min = (
    df["order_purchase_timestamp"]
    .min()
    .strftime("%b %Y")
)

purchase_max = (
    df["order_purchase_timestamp"]
    .max()
    .strftime("%b %Y")
)

purchase_window = f"{purchase_min} to {purchase_max}"

In [28]:
display(Markdown(f"""
**Dataset at a Glance**

| | |
|---|---|
| **Rows** | {n_rows} |
| **Features** | {n_features} |
| **Granularity** | Order-item level |
| **Order purchase window** | {purchase_window} |
| **Source** | `order_details.parquet` (output of *Data Preparation and Merging*) |
| **Marketplace entities** | Customers, sellers, products, orders, order items, payments, reviews, geolocation data, and product category translations |
| **Representative engineered features** | Logistics (`delivery_delay_days`, `customer_seller_distance_miles`), pricing (`freight_to_price_ratio`), seller profiling (`seller_type`, category concentration metrics), and product characteristics (`product_volume_cm3`) |
| **Note** | Outliers were retained. Missing delivery timestamps primarily correspond to valid order lifecycle states, including canceled, unavailable, and in-transit orders. |
"""))


**Dataset at a Glance**

| | |
|---|---|
| **Rows** | 113,425 |
| **Features** | 68 |
| **Granularity** | Order-item level |
| **Order purchase window** | Sep 2016 to Oct 2018 |
| **Source** | `order_details.parquet` (output of *Data Preparation and Merging*) |
| **Marketplace entities** | Customers, sellers, products, orders, order items, payments, reviews, geolocation data, and product category translations |
| **Representative engineered features** | Logistics (`delivery_delay_days`, `customer_seller_distance_miles`), pricing (`freight_to_price_ratio`), seller profiling (`seller_type`, category concentration metrics), and product characteristics (`product_volume_cm3`) |
| **Note** | Outliers were retained. Missing delivery timestamps primarily correspond to valid order lifecycle states, including canceled, unavailable, and in-transit orders. |
